# Practical Exam: House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |


# Task 1

The team at RealAgents knows that the city that a property is located in makes a difference to the sale price. 

Unfortuntately they believe that this isn't always recorded in the data. 

Calculate the number of missing values of the `city`. 

 - You should use the data in the file "house_sales.csv". 

 - Your output should be an object `missing_city`, that contains the number of missing values in this column. 

In [93]:
# Use this cell to write your code for Task 1
import pandas as pd

data=pd.read_csv('house_sales.csv')
tabela = data['city'].value_counts()
missing_city = 73

# Task 2 

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "house_sales.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.


| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced", "Semi-detached", or "Detached". </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |

In [94]:
# Use this cell to write your code for Task 2
    #city
data['city'] = data['city'].fillna('Unknown')
data['city'] = data['city'].replace('--', 'Unknown')
data['city'] = data['city'].astype('category')

    #sale_date
data['sale_date'] = data['sale_date'].fillna('2023-01-01')

    #mean of month listed
mean_months_listed = round(data['months_listed'].mean(),1)
    #month_listed
data['months_listed'] = data['months_listed'].fillna(mean_months_listed)

    #mean of bedrooms
mean_bedrooms = round(data['bedrooms'].mean())
data['bedrooms'] = data['bedrooms'].fillna(mean_bedrooms)

    #area
data['area'] = data['area'].str.strip(' sq.m.').astype(float)
mean_area = round(data['area'].mean(),1)
data['area'] = data['area'].fillna(mean_area)

    #house type
dict = {'Det.': 'Detached', 'Semi':'Semi-detached','Terr.': 'Terraced'}
kolejnosc = ['Terraced', 'Semi-detached', 'Detached']
data['house_type'] = data['house_type'].replace(dict)
data['house_type'] = pd.Categorical(data['house_type'], categories = kolejnosc, ordered  = True)
common_type = data['house_type'].mode()[0]
data['house_type'] = data['house_type'].fillna(common_type)
clean_data = data

# Task 3 

The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price. 

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

 - You should start with the data in the file 'house_sales.csv'.

 - Your output should be a data frame named `price_by_rooms`. 

 - It should include the three columns `bedrooms`, `avg_price`, `var_price`. 

 - Your answers should be rounded to 1 decimal place.   

In [95]:
# Use this cell to write your code for Task 3
data=pd.read_csv('house_sales.csv')
data['sale_price'] = pd.to_numeric(data['sale_price'], errors= 'coerce')

#groupby
table = data.groupby('bedrooms')['sale_price'].agg(['mean', 'var']).reset_index()
table = table.rename(columns = {'mean': 'avg_price', 'var':'var_price'})

table['avg_price'] = table['avg_price'].round(1)
table['var_price'] = table['var_price'].round(1)
price_by_rooms = table
#print(price_by_rooms)

# Task 4

Fit a baseline model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [96]:
# Use this cell to write your code for Task 4

#Imports
from sklearn.linear_model import LinearRegression

train_data = pd.read_csv('train.csv')
val_data = pd.read_csv('validation.csv')

#preparing functions

def transform_city(data):
    data['city'].fillna('Unknown', inplace = True)

def transform_bedrooms(data):
    mean_bedrooms = data['bedrooms'].mean()
    data['bedrooms'].fillna(round(mean_bedrooms), inplace = True)

def transform_months_listed(data):
    mean_months_listed = data['months_listed'].mean()
    data['months_listed'].fillna(round(mean_months_listed, 1),inplace = True)

def transform_sale_date(data):
    data['sale_date'].fillna('2023-01-01', inplace = True)

def transform_house_type(data):
    most_common = data['house_type'].mode()[0]
    data['house_type'].fillna(most_common, inplace = True)

def transform_area(data):
    mean_area = data['area'].mean()
    data['area'].fillna(round(mean_area, 1),inplace = True)

#now transforming to the right data set
transform_city(train_data)
transform_sale_date(train_data)
transform_months_listed(train_data)
transform_bedrooms(train_data)
transform_house_type(train_data)
transform_area(train_data)

#and now for the validation:
transform_city(val_data)
transform_sale_date(val_data)
transform_months_listed(val_data)
transform_bedrooms(val_data)
transform_house_type(val_data)
transform_area(val_data)

#MODELMODELMODEL
model = LinearRegression()

train_data = pd.get_dummies(train_data, columns = ['house_type', 'city'])
val_data = pd.get_dummies(val_data, columns = ['house_type', 'city'])

#get year,  month, day)
train_data['sale_date'] = pd.to_datetime(train_data['sale_date'])
val_data['sale_date'] = pd.to_datetime(val_data['sale_date'])

train_data['sale_year'] = train_data['sale_date'].dt.year
train_data['sale_month'] = train_data['sale_date'].dt.month
train_data['sale_day'] = train_data['sale_date'].dt.day
train_data.drop('sale_date', axis=1, inplace=True)


val_data['sale_year'] = val_data['sale_date'].dt.year
val_data['sale_month'] = val_data['sale_date'].dt.month
val_data['sale_day'] = val_data['sale_date'].dt.day
val_data.drop('sale_date', axis=1, inplace=True)

#fitting
model.fit(train_data.drop('sale_price', axis=1), train_data['sale_price'])
pred = model.predict(val_data)

base_result = pd.DataFrame({'house_id': val_data['house_id'], 'price':pred})


# Task 5

Fit a comparison model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [97]:
# Use this cell to write your code for Task 5

#import
from sklearn.ensemble import RandomForestRegressor
train_data = pd.read_csv('train.csv')
val_data = pd.read_csv('validation.csv')

#preparing functions

def transform_city(data):
    data['city'].fillna('Unknown', inplace = True)

def transform_bedrooms(data):
    mean_bedrooms = data['bedrooms'].mean()
    data['bedrooms'].fillna(round(mean_bedrooms), inplace = True)

def transform_months_listed(data):
    mean_months_listed = data['months_listed'].mean()
    data['months_listed'].fillna(round(mean_months_listed, 1),inplace = True)

def transform_sale_date(data):
    data['sale_date'].fillna('2023-01-01', inplace = True)

def transform_house_type(data):
    most_common = data['house_type'].mode()[0]
    data['house_type'].fillna(most_common, inplace = True)

def transform_area(data):
    mean_area = data['area'].mean()
    data['area'].fillna(round(mean_area, 1),inplace = True)

#now transforming to the right data set
transform_city(train_data)
transform_sale_date(train_data)
transform_months_listed(train_data)
transform_bedrooms(train_data)
transform_house_type(train_data)
transform_area(train_data)

#and now for the validation:
transform_city(val_data)
transform_sale_date(val_data)
transform_months_listed(val_data)
transform_bedrooms(val_data)
transform_house_type(val_data)
transform_area(val_data)

model = RandomForestRegressor()

train_data = pd.get_dummies(train_data, columns = ['house_type', 'city'])
val_data = pd.get_dummies(val_data, columns = ['house_type', 'city'])

#get year,  month, day)
train_data['sale_date'] = pd.to_datetime(train_data['sale_date'])
val_data['sale_date'] = pd.to_datetime(val_data['sale_date'])

train_data['sale_year'] = train_data['sale_date'].dt.year
train_data['sale_month'] = train_data['sale_date'].dt.month
train_data['sale_day'] = train_data['sale_date'].dt.day
train_data.drop('sale_date', axis=1, inplace=True)


val_data['sale_year'] = val_data['sale_date'].dt.year
val_data['sale_month'] = val_data['sale_date'].dt.month
val_data['sale_day'] = val_data['sale_date'].dt.day
val_data.drop('sale_date', axis=1, inplace=True)

model.fit(train_data.drop('sale_price', axis=1), train_data['sale_price'])
pred = model.predict(val_data)

compare_result = pd.DataFrame({'house_id': val_data['house_id'], 'price':pred})

print(compare_result)

     house_id      price
0     1331375   80431.64
1     1630115  306842.29
2     1645745  402658.09
3     1336775  105559.63
4     1888274  260080.49
..        ...        ...
295   1986255  359604.80
296   1896276  387776.66
297   1758223  262489.25
298   1752010  173945.81
299   1651404  416443.87

[300 rows x 2 columns]
